## 第 1 步：列表（list）和字典（dict）长什么样


In [6]:
# list 一串
names = ["张三","李四","王五"]

numbers = [1,2,3,4,5]

In [18]:
print(names)
print(numbers)

print(names[0])
print(numbers[0])

['张三', '李四', '王五']
[1, 2, 3, 4, 5]
张三
1
张三
李四
王五


In [8]:
# dict 键值对

person = {
    "name":"张三",
    "age":18,
    "gender":"男",
    "city":"北京"
}

In [14]:
print(person)

print(person["name"])

{'name': '张三', 'age': 18, 'gender': '男', 'city': '北京'}
张三


## 第 2 步：for 循环遍历

In [19]:
for name in names:
    print(name)

张三
李四
王五


#### 练习1.1：创建一个 bom_list 列表，里面放 3 个字典（bom_id、material、status）


In [22]:
bom_list = [
    {"bom_id": "BOM-001", "material": "螺丝 M3", "status": "待审核"},
    {"bom_id": "BOM-002", "material": "钣金件 A12", "status": "已通过"},
    {"bom_id": "BOM-003", "material": "电池 5000mAh", "status": "待审核"},
]

In [23]:
for bom in bom_list: 
    print(bom["bom_id"],bom["material"])

BOM-001 螺丝 M3
BOM-002 钣金件 A12
BOM-003 电池 5000mAh


## 第3步：if 条件判断

等于：==
不等于：!=
大于：> 小于：<

#### 练习1.2：结合循环和判断：找出 status 为“待审核”的 bom


In [26]:
for bom in bom_list:
    if bom["status"] == "待审核":
        print(bom["bom_id"], bom["material"])

BOM-001 螺丝 M3
BOM-003 电池 5000mAh


## 第 4 步：函数定义

In [28]:
def get_pending_bom(bom_list):
    result = []
    for bom in bom_list:
        if bom["status"] == "待审核":
            result.append(bom)
    return result

pending = get_pending_bom(bom_list)
for bom in pending:
    print(bom["bom_id"], bom)

## 第5步：上手 API 调用, 写第一个调模型的脚本

In [33]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


使用免费的 API： 
nvidia/nemotron-3.5-lightning:free

In [48]:
import requests
import json

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer [REDACTED]",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3.5-lightning:free",
    "messages": [
        {"role": "system", "content": "你是一个工单分类助手，只返回分类名称：权限/流程/接口/数据"},
        {"role": "user", "content": "我提交了报销单，财务说看不到"}
      ],
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
response = response['choices'][0]['message']

print(response)

{'role': 'assistant', 'content': '权限', 'refusal': None, 'reasoning': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "我提交了报销单，财务说看不到"\n   - Translation: "I submitted a reimbursement form, finance says they can\'t see it"\n   - Context: This seems like a system/ticket issue related to access/permissions or workflow\n\n2.  **Identify the Core Issue:**\n   - User submitted a reimbursement form\n   - Finance department can\'t see it\n   - This could be due to: permissions, workflow status, interface issues, or data problems\n\n3.  **Determine the Classification:**\n   - The prompt says: "你是一个工单分类助手，只返回分类名称：权限/流程/接口/数据"\n   - I need to output exactly one of: 权限, 流程, 接口, 数据\n   - Based on the issue: Finance can\'t see a submitted reimbursement form\n   - Possible categories:\n     - 权限 (Permissions): Maybe user doesn\'t have right access, or finance doesn\'t have permission to view certain submissions\n     - 流程 (Process): Maybe the form isn\'t routed correctly, 

In [49]:
import requests
import json

message = [
        {"role": "system", "content": "你是一个工单分类助手，只返回分类名称：权限/流程/接口/数据"},
        {"role": "user", "content": "我提交了报销单，财务说看不到"}
      ]

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer [REDACTED]",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3.5-lightning:free",
    "messages": message,
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
response = response['choices'][0]['message']

print(response)

{'role': 'assistant', 'content': 'Permission', 'refusal': None, 'reasoning': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "我提交了报销单，财务说看不到" \n   - Translation: "I submitted a reimbursement form, finance says they can\'t see it"\n   - Context: This is about a ticket/classification system for IT/service desk or internal requests.\n   - The prompt says: "You are a ticket classification assistant, only return the category name: Permission/Process/Interface/Data"\n\n2.  **Determine the Category:**\n   - The issue is about submitting a reimbursement form, and finance can\'t see it.\n   - This relates to access rights, permissions, or system access.\n   - Looking at the allowed categories: Permission/Process/Interface/Data\n   - "Permission" seems most appropriate: Finance doesn\'t have permission/access to see the reimbursement form.\n   - Could be "Process" if it\'s about workflow, but the core issue is "can\'t see" which implies access/permission.\n   - "Inte

## 第 6 步：改成交互式输入

In [51]:
import requests
import json


# 用户人工输入
ticket_message = input("请输入工单描述：") 
# 我申请了加班，主管说没看到 
# ERP 系统打不开

message = [
        {"role": "system", "content": "你是一个工单分类助手，只返回分类名称：权限/流程/接口/数据"},
        {"role": "user", "content": ticket_message}
      ]

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer [REDACTED]",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3.5-lightning:free",
    "messages": message,
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
response = response['choices'][0]['message']['content']

print(response)

请输入工单描述：我申请了加班，主管说没看到
权限


## 第 7 步：加一个函数包装

In [55]:
import requests
import json

def classify_ticket(ticket_message):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers={
    "Authorization": "Bearer [REDACTED]",
    "Content-Type": "application/json",
  }
    data=json.dumps({
    "model": "nvidia/nemotron-3.5-lightning:free",
    "messages": [
        {"role": "system", "content": "你是一个工单分类助手，只返回分类名称：权限/流程/接口/数据"},
        {"role": "user", "content": ticket_message}
      ],
    "reasoning": {"enabled": True}
  })
    response = requests.post(url=url,headers=headers,data=data)
    response = response.json()['choices'][0]['message']['content']
    return response

# 用户人工输入
ticket_message=input("请输入工单描述：") # ERP 系统打不开
result=classify_ticket(ticket_message)
print(result)

请输入工单描述：ERP 系统打不开
权限
